# The front end over every training photo

The species classifier trains on photos cut down to the snake, using the box from a detector (and for some variants
the mask from a segmenter). The cropping notebook (13) ran those two models over about 39,500 photos: the experiment
subset, the validation photos, the calibration photos and half of the unknown pool. The full-data run trains on all
49,782 photos left after the calibration photographers are set aside, so 38,853 of them have no box yet.

This notebook runs the same two models, with the same weights, over exactly those missing photos and joins the result
to the earlier file. It never retrains anything. The rows from notebook 13 are kept byte for byte; only new rows are
added after them.

*Note added after the runs:* this notebook was not run on its own. A Kaggle GPU session could wait hours in
the queue, so notebook 21 runs the same step, `front_end.extend_predictions` with notebook 13's weights,
inside each full-training session just before training (its section 4, step 1). The results of those runs are
under `results/m4/full/`, and their executed copies are in `results/m4/full/executed/`.

In [ ]:
%pip install -q albumentations==2.0.8 timm==1.0.31 scikit-learn==1.9.1
%pip install -q ultralytics==8.4.202 segmentation-models-pytorch==0.6.0

import hashlib, json, os, shutil, sys, time, warnings
from pathlib import Path

os.environ.update(NO_ALBUMENTATIONS_UPDATE="1", YOLO_VERBOSE="False", HF_HUB_DISABLE_PROGRESS_BARS="1")
warnings.filterwarnings("ignore")

import cv2, kagglehub, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Dataset
from ultralytics import YOLO

SRC = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(SRC if SRC.exists() else Path("../src").resolve()))
from serpentguard import data
from serpentguard.data import clean_mask, load_mask, load_rgb
from serpentguard.metrics import TEST_SPLITS
from serpentguard.front_end import BOX_FLOOR, DEVICE, WORKERS, Segmenter, extend_predictions, snake_boxes

LIMIT = None  # a number runs only that many new photos (a smoke test); None runs them all
FRONT_DATASETS = None  # None reads notebook 13's kernel output; else [weights dataset, predictions dataset] handles
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("../work/front_end_full")
PREDS = OUT / "frontend_preds"
CHECKS = OUT / "checks"
for folder in (PREDS, CHECKS):
    folder.mkdir(parents=True, exist_ok=True)
CHECK_ROWS = 20            # photos looked at in each check
M3_ROWS = 39_526           # rows in notebook 13's frontend_preds.csv
FULL_ROWS, FULL_CLASSES = 49_782, 184
M3_CSV_SHA256 = "9dcc0576eb1c60d3e767ba9ca3c34a6a1b92b6b01946e377042dee1b1287b9e3"  # notebook 13's frontend_preds.csv
DETECTOR_SHA256 = "099b36bd8271c328b60be8152161f74558a0083ebe4e6fec1f558e7c0951c5b0"  # m3-video-demo/models/detector.pt
SUBSET_FINGERPRINT = "40196d19a68e"
print(f"{DEVICE}, {torch.cuda.device_count()} GPU(s), {WORKERS} CPUs; results go to {OUT}")

## 1. Which photos need a box

The list is built from the project's own functions, so it cannot drift from the full-data run: the training photos,
minus the calibration photographers' photos, capped at 50 per photographer and 1,000 per species. It must come to
49,782 photos over 184 species. The list is saved as `full_train.csv`.

In [ ]:
def dataset(slug):
    return Path(kagglehub.dataset_download(f"puneetbajajiitm/{slug}"))

DATA = dataset("serpentguard-indian-snakes-clean")
ids = {"obs_id": str, "photo_id": str, "observer_id": str}
metadata = pd.read_csv(DATA / "metadata.csv", dtype=ids, low_memory=False)
subset = pd.read_csv(dataset("serpentguard-m3-subset") / "subset.csv", dtype=ids)
fingerprint = hashlib.sha256("\n".join(sorted(subset["path"])).encode()).hexdigest()
assert fingerprint.startswith(SUBSET_FINGERPRINT), "this is not the experiment subset"

train = metadata[metadata["split"] == "train"]
calibration = data.calibration_set(train, subset)
rows = train[~train["observer_id"].isin(calibration["observer_id"])]
full = data.cap_train(rows)
assert len(full) == FULL_ROWS and full["species"].nunique() == FULL_CLASSES, (len(full), full["species"].nunique())
full[["path"]].to_csv(PREDS / "full_train.csv", index=False)
print(f"{len(full):,} photos over {full['species'].nunique()} species; list saved as full_train.csv")

## 2. The notebook 13 models

The weights come from the saved output of the notebook 13 run (kernel `sg-m3-frontends`, or a dataset copy of the same files when the run is on another account; the prediction file's SHA-256 is checked too): the detector is the
fine-tuned YOLO26n, the segmenter the U-Net that won on the human outlines. The detector's SHA-256 is compared with
the copy kept on the laptop for the video demo, so we know it is the very file the demo and the earlier
`frontend_preds.csv` came from.

In [ ]:
if FRONT_DATASETS is None:
    WEIGHTS_ROOT = PREDS_ROOT = Path(kagglehub.notebook_output_download("puneetbajajiitm/sg-m3-frontends"))
else:  # the same files, copied to datasets that another account can read
    WEIGHTS_ROOT, PREDS_ROOT = (Path(kagglehub.dataset_download(handle)) for handle in FRONT_DATASETS)
DETECTOR = next(WEIGHTS_ROOT.rglob("D1/yolo26n-ft/detector.pt"))
UNET = next(WEIGHTS_ROOT.rglob("S1/unet/segmenter.pt"))
M3_PREDS = next(PREDS_ROOT.rglob("frontend_preds.csv")).parent
assert hashlib.sha256((M3_PREDS / "frontend_preds.csv").read_bytes()).hexdigest() == M3_CSV_SHA256,     "frontend_preds.csv is not notebook 13's file"
sha = hashlib.sha256(DETECTOR.read_bytes()).hexdigest()
assert sha == DETECTOR_SHA256, f"detector differs from the laptop copy: {sha}"
print("detector sha256", sha, "(same as m3-video-demo/models/detector.pt)")
detector = YOLO(str(DETECTOR))
seg = Segmenter("unet", UNET.parent)
m3 = pd.read_csv(M3_PREDS / "frontend_preds.csv", keep_default_na=False)
assert len(m3) == M3_ROWS and m3["path"].is_unique, len(m3)
print(f"{len(m3):,} rows in notebook 13's frontend_preds.csv")

## 3. Check that the models still reproduce notebook 13

Before running anything new, 20 photos that notebook 13 already handled are run again, 10 that have a box and 10 that
do not (or fewer, if few have none). Every box must agree within 1 pixel and every confidence within 0.001; a photo
that had no box must still have none. If this fails the weights or the code are not the ones that made the earlier
file, and nothing below can be trusted. The photos live on Kaggle, so this check runs here, not on the laptop.

In [ ]:
def predict(paths):
    """(box or None, confidence or None, mask or None) per path, exactly as notebook 13 chose them."""
    rgbs = [load_rgb(str(DATA / "data" / p)) for p in paths]
    found = snake_boxes(detector, [cv2.cvtColor(r, cv2.COLOR_RGB2BGR) for r in rgbs], conf=BOX_FLOOR)
    masks = seg.masks(rgbs)
    result = []
    for mask, candidates in zip(masks, found):
        best = max(candidates, key=lambda b: b[4], default=None)
        result.append((best[:4] if best else None, best[4] if best else None, mask))
    return result

has_box = m3["box"].ne("")
no_box = m3[~has_box].head(CHECK_ROWS // 2)
rng_sample = pd.concat([m3[has_box].sample(CHECK_ROWS - len(no_box), random_state=42), no_box])
worst_box, worst_conf, mask_agree = 0.0, 0.0, 0
for row, (box, conf, mask) in zip(rng_sample.itertuples(), predict(list(rng_sample["path"]))):
    if row.box == "":
        assert box is None, f"{row.path}: notebook 13 found no box, the re-run found one"
    else:
        old = json.loads(row.box)
        assert box is not None, f"{row.path}: the re-run found no box"
        worst_box = max(worst_box, float(np.abs(np.array(box) - np.array(old)).max()))
        worst_conf = max(worst_conf, abs(conf - float(row.box_conf)))
    mask_agree += (clean_mask(mask) is not None) == (row.mask_file != "")
assert worst_box <= 1.0, f"a box moved by {worst_box:.2f} px"
assert worst_conf <= 0.001, f"a confidence moved by {worst_conf:.5f}"
assert mask_agree == len(rng_sample), f"a mask appeared or vanished on {len(rng_sample) - mask_agree} photos"
print(f"{len(rng_sample)} photos re-predicted: largest box difference {worst_box:.3f} px, "
      f"largest confidence difference {worst_conf:.5f}; every mask is kept or dropped as before")

## 4. Run the models over the missing photos and join the result

`front_end.extend_predictions` does the work, so the full-training notebook can run the very same step. Only photos in
`full_train.csv` that are not already in notebook 13's file are run. For each, the detector's most confident box at
0.05 or above becomes the `box` (with its `box_conf`), and the U-Net's mask is saved as a PNG under `masks_pred/` only
if the cleanup keeps it; otherwise the field stays empty. Each batch is appended to a file as it finishes, so a
restarted run carries on. The final `frontend_preds.csv` is notebook 13's file, unchanged, followed by the new rows;
notebook 13's masks are copied beside the new ones so that every `mask_file` resolves inside this one folder.

In [ ]:
missing = full[~full["path"].isin(m3["path"])][["path"]]
if LIMIT:
    missing = missing.head(LIMIT)
m3_text = (M3_PREDS / "frontend_preds.csv").read_bytes()
final = extend_predictions(full, M3_PREDS, DETECTOR, UNET, DATA, PREDS, limit=LIMIT)
new_rows = final.iloc[M3_ROWS:]
print(f"{len(final):,} rows = {M3_ROWS:,} from notebook 13 + {len(new_rows):,} new")

## 5. Read the result back

A file that was written quietly can still be wrong, so it is read back and checked against sources that did not
produce it: the photo list, notebook 13's own file, and the photos themselves.

1. The row count is notebook 13's 39,526 plus the new rows, with no photo twice, and every photo of `full_train.csv` has a row.
2. The first 39,526 rows are byte for byte notebook 13's.
3. A negative control: no test-split photo is in the file (the control looks for photos that exist, so a pass means something).
4. Every row with a box has a confidence, and no row has a confidence without a box.
5. 20 random new rows are drawn on their photos and saved as a picture sheet, to be looked at by eye; 20 new masks are loaded at their photo's size.

In [ ]:
expected = set(m3["path"]) | set(missing["path"])
assert len(final) == M3_ROWS + len(new_rows) and final["path"].is_unique, "row count or duplicates"
assert set(final["path"]) == expected, "the rows are not notebook 13's photos plus the missing ones"
assert LIMIT or set(full["path"]) <= set(final["path"]), "a training photo has no row"
assert LIMIT or len(new_rows) == len(missing), "new rows and missing photos differ in number"

assert (PREDS / "frontend_preds.csv").read_bytes().startswith(m3_text), "notebook 13's rows changed"

test_paths = set(metadata.loc[metadata["split"].isin(TEST_SPLITS), "path"])
assert test_paths, "the negative control needs test photos to look for"
assert not test_paths & set(final["path"]), "a test-split photo has a row"

box_present = final["box"].ne("")
assert box_present.equals(final["box_conf"].ne("")), "a box without its confidence or the reverse"
assert pd.to_numeric(final.loc[box_present, "box_conf"]).notna().all(), "NaN confidence where a box exists"
assert pd.to_numeric(final.loc[box_present, "box_conf"]).ge(BOX_FLOOR).all(), "a box under the floor"
print(f"{len(final):,} rows, no duplicates, notebook 13's {M3_ROWS:,} rows byte-identical, "
      f"{(~box_present[M3_ROWS:]).sum()} new rows without a box")

In [ ]:
sample = new_rows[new_rows["box"].ne("")].sample(min(CHECK_ROWS, int(new_rows["box"].ne("").sum())), random_state=42)
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax in axes.flat:
    ax.axis("off")
for ax, row in zip(axes.flat, sample.itertuples()):
    rgb = load_rgb(str(DATA / "data" / row.path))
    h, w = rgb.shape[:2]
    x1, y1, x2, y2 = json.loads(row.box)
    assert 0 <= x1 < x2 <= w + 1 and 0 <= y1 < y2 <= h + 1, f"box {row.box} is outside {row.path}"
    shade = rgb.copy()
    if row.mask_file:
        on = load_mask(str(PREDS / row.mask_file), (h, w)) == 1
        shade[on] = (0.6 * shade[on] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)
    ax.imshow(shade)
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="red", lw=2))
    ax.set_title(f"{Path(row.path).parent.name} conf {row.box_conf}", fontsize=9)
plt.tight_layout()
fig.savefig(CHECKS / "overlay_sheet.png", dpi=60)
plt.close(fig)
masked = new_rows[new_rows["mask_file"].ne("")].sample(min(CHECK_ROWS, int(new_rows["mask_file"].ne("").sum())),
                                                        random_state=42)
for path, mask_file in masked[["path", "mask_file"]].values:
    shape = load_rgb(str(DATA / "data" / path)).shape[:2]
    assert load_mask(str(PREDS / mask_file), shape) is not None, f"{mask_file} fails load_mask"
print(f"{len(sample)} boxes inside their photos and drawn on checks/overlay_sheet.png (photos stay out of this notebook); "
      f"{len(masked)} masks load at their photo's size")

## 6. How many photos have a box

The same coverage numbers notebook 13 printed, for the new photos, so the two can be compared. A box means the
detector was at least 0.05 sure; "at 0.25" counts only the confident ones, the cut the classifier later learns for itself.

In [ ]:
conf = pd.to_numeric(new_rows["box_conf"], errors="coerce")
coverage = {"photos": len(new_rows), "box": float(new_rows["box"].ne("").mean()),
            "box_at_conf": float((conf >= 0.25).mean()), "mask": float(new_rows["mask_file"].ne("").mean())}
summary = {"photos": len(final), "new_photos": len(new_rows), "m3_photos": M3_ROWS, "box_floor": BOX_FLOOR,
           "detector_sha256": sha, "segmenter": "unet", "new_coverage": coverage,
           "limit": LIMIT}
(PREDS / "summary.json").write_text(json.dumps(summary))
pd.Series(coverage).round(3).to_frame("new photos")

## What we found

This notebook has no outputs, since it was not run on its own (see the note at the top). In each of notebook
21's three full runs the same step added 38,853 new photos to notebook 13's 39,526 rows, 78,379 rows in all,
with a box for 100.0% of the new photos and a mask for 94.3% (the front-end line of the full pass in
`results/m4/full/executed/21_full_training_<run>.ipynb`; the row count is also in each run's `manifest.json`).